In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请描述这张图的内容"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://api.nga.gov/iiif/a2e6da57-3cd1-4235-b20e-95dcaefed6c8/full/!800,800/0/default.jpg",
                    }
                },
            ],
        }
    ],
    max_tokens=300,
)

print(response.choices[0].message.content)

In [ ]:
import base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Path to your image
image_path = "E:\\llm_workspace\\online_workspace\\llm_quickstart\\api-quickstart\\03-多模态\\james.jpg"

# Getting the Base64 string
base64_image = encode_image(image_path)


response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请描述这张图的内容"},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
    max_tokens=300,
)

print(response.choices[0].message.content)

In [ ]:
from PIL import Image
import io
import base64

def compress_image(image_path, max_size=(800, 800)):
    """压缩图片到合适大小"""
    with Image.open(image_path) as img:
        img.thumbnail(max_size)
        
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=85)
        
        return base64.b64encode(buffer.getvalue()).decode('utf-8')

base64_image = compress_image("aaa.jpg")
print(f"压缩后大小: {len(base64_image)/1024:.2f} KB")  # 确保 <500KB

In [ ]:
messages=[{
    "role": "user",
    "content": [
        {"type": "text", "text": "请描述这张图片的内容"},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
    ]
}]

response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages,
    max_tokens=100
)

print(f"AI回复：{response.choices[0].message.content}")

In [ ]:
response = client.chat.completions.create(
  model="google/gemini-3-pro-image-preview",
  messages=[
          {
            "role": "user",
            "content": "请生成一张勒布朗詹姆斯头戴皇冠的照片"
          }
        ],
  extra_body={"modalities": ["image", "text"]}
)

response = response.choices[0].message
if response.images:
  for image in response.images:
    image_url = image['image_url']['url'] 
    print(f"Generated image: {image_url}...")

In [ ]:
from IPython.display import Image, display
display(Image(url=image_url))